In [1]:
import pandas as pd
import numpy as np
import scipy.stats
from scipy import stats
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

from matplotlib import pyplot as plt
from matplotlib_venn import venn3

import statsmodels.api as sm

import seaborn as sns
import descri_function as des_fun

# Read data

In [13]:
# dado com os modelos 

df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/aesop_03_04_2026_with_MEM_all_mods_ens.parquet')

dta = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/cities_valid_for_MEM_26_03_2026.parquet')

In [14]:
# Select cities for the manuscript analysis (valid MEM)
lst = list(set(df.co_ibge.unique()) - set(dta.co_ibge.unique()))
df = df[~df.co_ibge.isin(lst)]

df =  df[(df.year_week >= '2022-42') & (df.year_week <= '2025-32')]

# Columnas que nos interessa

In [15]:
# uma coluna identificando as regioes (aqui municipio), a coluna com o resultado do modelo (0/1) vou usa aqui a do EARS,
# a coluna que queremos antecipar vou supor que seja a do EVI.

df = df[['co_ibge', 'year_week','sinal_ears_atend', 'sinal_evi_ivas']] 

In [16]:
# Primeiro precisa limpar a coluna que queremos antecipar (evi), agrupando os avisos  e identificando a semana onde o surto começa

df = des_fun.clean_warning_column(df, 'co_ibge','year_week','sinal_evi_ivas')

In [17]:
df = df[['co_ibge', 'year_week', 'sinal_ears_atend', 'sinal_evi_ivas',  'cleaned_warning', 
       'event',  'warning_final']]

In [18]:
df = df.rename(columns={'cleaned_warning': 'warning_evi_without_isolated', 
                            'event': 'warning_evi_corect_with_consec',
                           'warning_final':'warning_final_evi'})

In [19]:
df

,co_ibge,year_week,sinal_ears_atend,sinal_evi_ivas,warning_evi_without_isolated,warning_evi_corect_with_consec,warning_final_evi
0,110001,2022-42,1,0,0,0,0
6827,110001,2022-43,0,0,0,0,0
12577,110001,2022-44,0,0,0,0,0
18361,110001,2022-45,1,0,0,0,0
24654,110001,2022-46,1,1,0,0,0
...,...,...,...,...,...,...,...
764648,412560,2025-28,0,0,0,0,0
769583,412560,2025-29,0,0,0,0,0
775456,412560,2025-30,0,0,0,0,0
780503,412560,2025-31,0,0,0,0,0


# Get peformance


In [20]:

df_warning_count = des_fun.antici_count(df, 'sinal_ears_atend', 'warning_final_evi', 'warning_evi_corect_with_consec', 'co_ibge')
    
performance_summary = des_fun.summarize_performance(df_warning_count)


In [21]:
# Dado base utilizado para calcular a performance
df_warning_count # a coluna total_aih_warning representa total_evi_warning na região

,co_ibge,n3,n2,n1,n0,n1_after,missed,TP,TP_,TN1,FP1,FN,total_aih_warning
0,110001,2,0,0,1,0,0,3,15,112,20,0,3
1,354830,0,0,0,0,0,0,0,0,106,41,0,0
2,280030,0,0,0,2,1,2,2,14,91,36,2,4
3,251140,2,1,0,0,0,2,3,20,88,33,2,5
4,421570,2,0,0,1,0,0,3,19,108,20,0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5360,261090,0,0,1,2,0,0,3,19,110,18,0,3
5361,353390,0,0,1,6,1,4,7,47,75,13,4,11
5362,314160,0,0,0,1,1,2,1,9,100,32,2,3
5363,315010,1,0,1,5,0,1,7,41,76,27,1,8


In [22]:
performance_summary

,Metric,Value
0,Total Warnings,19176
1,Early Detection (1-3 weeks),7237 (37.7%)
2,Timely Detection (0 weeks),7236 (37.7%)
3,Missed Warnings,4703 (24.5%)
4,Sensitivity,75.5%
5,Specificity,80.4%
6,PPV,40.8%
7,NPV,99.2%
8,POD,86.7%
9,FPR,19.6%
